In [1]:
!pip install dagshub

In [3]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
import sklearn
import numpy as np
import dagshub
dagshub.init(repo_owner='slomi23', repo_name='slomi23_ML_Assignment_1', mlflow=True)
import os
#helper funciton
def print_mean_and_std(train_scores, val_scores):
    print("train_scores mean:", np.mean(train_scores))
    print("train_scores std:", np.std(train_scores))
    print("val_scores mean:", np.mean(val_scores))
    print("val_scores std:", np.std(val_scores))


Initialized MLflow to track repo "slomi23/slomi23_ML_Assignment_1"

Repository slomi23/slomi23_ML_Assignment_1 initialized!

In [6]:
import requests
response = requests.get(
    "https://dagshub.com/api/v1/user",
    auth=("slomi23", "4eb3d59602f03f11e662743efeeb16d46ec11d50")
)
print(response.status_code)
print(response.json())

200
{'id': 95351, 'login': 'slomi23', 'full_name': 'Sandro Lomidze', 'avatar_url': 'https://dagshub.com/avatars/95351', 'public_email': '', 'website': '', 'company': '', 'description': '', 'username': 'slomi23'}


In [7]:
# read the train set
trainset=pd.read_csv("house-prices-advanced-regression-techniques/train.csv")
trainset["SalePrice"] = np.log1p(trainset["SalePrice"])
trainsetY=trainset["SalePrice"]
trainsetX=trainset.drop(columns=["SalePrice"])
# read the test set
testset=pd.read_csv("house-prices-advanced-regression-techniques/test.csv")
#trainset.head(100)
trainset.shape


(1460, 81)

<h>Filling null values</h1>


In [8]:
# sort it with non null value counts
print(trainsetX.count().sort_values(ascending=True).to_string())

PoolQC              7
MiscFeature        54
Alley              91
Fence             281
MasVnrType        588
FireplaceQu       770
LotFrontage      1201
GarageType       1379
GarageCond       1379
GarageFinish     1379
GarageYrBlt      1379
GarageQual       1379
BsmtExposure     1422
BsmtFinType2     1422
BsmtFinType1     1423
BsmtCond         1423
BsmtQual         1423
MasVnrArea       1452
Electrical       1459
Neighborhood     1460
BldgType         1460
Condition2       1460
Condition1       1460
Id               1460
LandContour      1460
Utilities        1460
LotConfig        1460
LandSlope        1460
HouseStyle       1460
OverallQual      1460
OverallCond      1460
MSSubClass       1460
ExterCond        1460
Foundation       1460
Exterior2nd      1460
ExterQual        1460
YearRemodAdd     1460
RoofStyle        1460
RoofMatl         1460
Exterior1st      1460
YearBuilt        1460
Heating          1460
CentralAir       1460
HeatingQC        1460
BsmtFinSF2       1460
BsmtUnfSF 

checking the features with most nulls, if null has a meaning (spoiler: it does here)

In [9]:
trainsetX["PoolQC"].unique()
# nan means no pool, which shouldnt be dropped 
# since its important for price
trainsetX.fillna({"PoolQC":"No Pool"}, inplace=True)

trainsetX["Fence"].unique()
#nan means no fence, which shouldnt be dropped 
# since it's important for price
trainsetX.fillna({"Fence":"No Fence"}, inplace=True)

trainsetX["MiscFeature"].unique()
trainsetX.fillna({"MiscFeature":"None"}, inplace=True)

trainsetX["Alley"].unique()
trainsetX.fillna({"Alley":"No Alley"}, inplace=True)

trainsetX["FireplaceQu"].unique()
trainsetX.fillna({"FireplaceQu":"No Fireplace"}, inplace=True)

trainsetX["MasVnrType"].unique()
trainsetX.fillna({"MasVnrType":"None"}, inplace=True)


#drop columns where mode is null
for col in trainsetX.columns:
    if trainsetX[col].mode()[0] == np.nan:
        trainsetX.drop(col, axis=1, inplace=True)


first, lets try simple devision of training set and validation set 80:20. (later we'll do kfold)

In [10]:
from sklearn.model_selection import train_test_split




first lets convert every categorical feature into numerical for linear regression, simply, lets assign each category to the mean of the target when grouped by that category

In [11]:
from sklearn.preprocessing import StandardScaler

# function to fill nas in both categorical and numerical columns
def fill_nas(data_to_change, data_to_read):
    cat_cols = data_to_change.select_dtypes(include="object").columns

    for col in cat_cols:
        data_to_change[col]=data_to_change[col].fillna(data_to_read[col].mode()[0])
    #now numerical columns
    num_cols = data_to_change.select_dtypes(exclude="object").columns
    for col in num_cols:
        data_to_change[col] = data_to_change[col].fillna(data_to_read[col].mean())
    return data_to_change
# func to turn categorical columns to numerical using mean encoding based on SalePrice mean
def cat_to_num_func(data_to_change, data_for_mapping):
    global_mean = data_for_mapping["SalePrice"].mean()
    cat_to_num_maps = {}
    cat_cols = data_to_change.select_dtypes(include="object").columns
    for col in cat_cols:
        cat_to_num_maps[col] = data_for_mapping.groupby(col)["SalePrice"].mean()


    for col in cat_cols:
        data_to_change[col + "_num"] = data_to_change[col].map(cat_to_num_maps[col]).fillna(global_mean)
        data_to_change.drop(col, axis=1, inplace=True)  
    return data_to_change

# func to scale data
def data_scale(data_to_change, scaler=None):
    if scaler is None:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(data_to_change)
    else:
        X_scaled = scaler.transform(data_to_change)
    return X_scaled, scaler
def preprocess (data_to_change, data_for_mapping, scaler=None):
    data_to_change=pd.DataFrame(data_to_change.copy())
    data_for_mapping=pd.DataFrame(data_for_mapping.copy())
    data_to_change = fill_nas(data_to_change, data_for_mapping)
    data_to_change = cat_to_num_func(data_to_change, data_for_mapping)

    # drop Id and SalePrice if they exist
    for col in ["Id", "SalePrice"]:
        if col in data_to_change.columns:
            data_to_change.drop(columns=[col], inplace=True)
    col_names = data_to_change.columns.tolist()
    X_scaled, scaler = data_scale(data_to_change, scaler)
    return pd.DataFrame(X_scaled, columns=col_names), scaler


In [12]:

from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

  
# func to do RFE and return only selected number of features
def data_rfe(data_to_change, data_to_fit, y, number_of_features=20):
    data_to_fit = pd.DataFrame(data_to_fit)
    data_to_change = pd.DataFrame(data_to_change)
    modelLR = LinearRegression()
    rfe=RFE(modelLR, n_features_to_select=number_of_features, step=0.1)
    rfe.fit(data_to_fit, y)
    
    rfe_selected_features = data_to_fit.columns[rfe.support_].to_list()
    #for i, feature in enumerate(rfe_selected_features, 1):
        #print(f"{i}. {feature}")
    data_to_change_rfe = data_to_change[rfe_selected_features]
    return data_to_change_rfe




X_train_LR is ready, now lets do mlflow log on simple linear regression

In [13]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error
"""mlflow.set_tracking_uri(
    "sqlite:////home/sandro/slomi23_ML_Assignment_1/mlflow.db"
)"""
#set up mlflow stuff
mlflow.set_tracking_uri("https://dagshub.com/slomi23/slomi23_ML_Assignment_1.mlflow")
mlflow.set_experiment("House Price Prediction")


with mlflow.start_run(run_name="linear_regression_without_OHE_and_kfold_final"):
    X=trainsetX.copy()
    y = trainsetY.copy()
    X_train_LR, X_val_LR, y_train_LR, y_val_LR = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )
    X_train_LR["SalePrice"] = y_train_LR
    X_val_LR["SalePrice"] = y_val_LR
    X_train_raw = X_train_LR.copy() 
    X_train_LR, scaler1 = preprocess(X_train_LR, X_train_LR)
    X_val_LR, _ = preprocess(X_val_LR, X_train_raw, scaler1)
    X_train_LR_selected= data_rfe(X_train_LR, X_train_LR, y_train_LR, 15)
    X_val_LR_selected= data_rfe(X_val_LR, X_train_LR, y_train_LR, 15)
    X_val_LR_selected=fill_nas(X_val_LR_selected, X_train_LR_selected)
    
    model = LinearRegression()
    model.fit(X_train_LR_selected, y_train_LR)

    # predictions
    train_preds = model.predict(X_train_LR_selected)
    val_preds = model.predict(X_val_LR_selected)
    #rmse
    train_rmse = np.sqrt(mean_squared_error(y_train_LR, train_preds))
    val_rmse = np.sqrt(mean_squared_error(y_val_LR, val_preds))

    # log metrics an praams
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("scaler", "StandardScaler")
    mlflow.log_params(model.get_params())
    mlflow.log_param("n_features", X_train_LR_selected.shape[1])
    print(f"Train RMSE: {train_rmse}")
    print(f"Val RMSE: {val_rmse}")

    final_model1 = LinearRegression()
    X["SalePrice"] = y
    processed_data, _ = preprocess(X, X)
    selected_data = data_rfe(processed_data, processed_data, y, number_of_features=15)
    selected_cols = selected_data.columns.tolist()
    pd.Series(selected_cols).to_csv("selected_features.csv", index=False)    
    final_model1.fit(selected_data, y)
    preds=final_model1.predict(selected_data)
    final_rmse = np.sqrt(mean_squared_error(y, preds))
    mlflow.log_metric("final_rmse", final_rmse)
    print(f"Final RMSE on full data: {final_rmse}")
    # log model
    mlflow.sklearn.log_model(final_model1, "linear_regression_without_OHE_and_kfold_final")


Train RMSE: 0.13693953512031173
Val RMSE: 0.14338270136704678


2026/04/12 11:11:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Final RMSE on full data: 0.13460983144518435


2026/04/12 11:12:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run linear_regression_without_OHE_and_kfold_final at: https://dagshub.com/slomi23/slomi23_ML_Assignment_1.mlflow/#/experiments/0/runs/e4da69816efd48dbb0f403fc4455bb14
🧪 View experiment at: https://dagshub.com/slomi23/slomi23_ML_Assignment_1.mlflow/#/experiments/0


Train log RMSE: 0.13
Val log RMSE: 0.143
now lets try kfold. still without OHE

In [14]:

#make train data for linerar regression with kfold
X_train_LR_KF = trainsetX.copy()
X_train_LR_KF["SalePrice"] = trainsetY
y_train_LR_KF = trainsetY

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
train_rmse_scores = []
val_rmse_scores = []
with mlflow.start_run(run_name="linear_regression_with_kfold"):
    for train_idx, val_idx in kfold.split(X_train_LR_KF):
        X_train1, X_val = X_train_LR_KF.iloc[train_idx].copy(), X_train_LR_KF.iloc[val_idx].copy()
        y_train1, y_val = y_train_LR_KF.iloc[train_idx], y_train_LR_KF.iloc[val_idx]
        X_train_raw = X_train1.copy()
        X_train1 = preprocess(X_train1, X_train1)[0]
        X_val = preprocess(X_val, X_train_raw, scaler1)[0]
        X_train1_raw = X_train1.copy()
        X_train1 = data_rfe(X_train1, X_train1, y_train1, number_of_features=15)
        X_val = data_rfe(X_val, X_train1_raw, y_train1, number_of_features=15)
        
        model = LinearRegression()
        model.fit(X_train1, y_train1)
        
        #predictions and rmse
        preds = model.predict(X_val)
        train_rmse= np.sqrt(mean_squared_error(y_train1, model.predict(X_train1)))
        val_rmse = np.sqrt(mean_squared_error(y_val, preds))
        train_rmse_scores.append(train_rmse)
        val_rmse_scores.append(val_rmse)

        #log metric and params
        mlflow.log_metric(f"train_rmse_fold_{fold}", train_rmse)
        mlflow.log_metric(f"val_rmse_fold_{fold}", val_rmse)
        mlflow.log_param("model_type", "LinearRegression")
        mlflow.log_param("scaler", "StandardScaler")
        mlflow.log_params(model.get_params())
        print(f"Fold {fold} - Train RMSE: {train_rmse}")
        print(f"Fold {fold} - Val RMSE: {val_rmse}")
        fold += 1

    mlflow.log_metric("val_rmse_mean", np.mean(val_rmse_scores))
    mlflow.log_metric("val_rmse_std", np.std(val_rmse_scores))
    mlflow.log_metric("train_rmse_mean", np.mean(train_rmse_scores))
    mlflow.log_metric("train_rmse_std", np.std(train_rmse_scores))


    final_model = LinearRegression()
    processed_data=preprocess(X_train_LR_KF, X_train_LR_KF)
    final_model.fit(processed_data[0], y_train_LR_KF)
    preds=final_model.predict(processed_data[0])
    final_rmse= np.sqrt(mean_squared_error(y_train_LR_KF, preds))
    mlflow.log_metric("final_rmse", final_rmse)
    print(f"Final RMSE: {final_rmse}")
    print_mean_and_std(train_rmse_scores, val_rmse_scores)
    mlflow.sklearn.log_model(final_model, "linear_regression_with_kfold")

    

Fold 1 - Train RMSE: 0.13693953512031173
Fold 1 - Val RMSE: 0.1433827013670468
Fold 2 - Train RMSE: 0.13639506367512416
Fold 2 - Val RMSE: 0.13708684219928605
Fold 3 - Train RMSE: 0.11708515639711824
Fold 3 - Val RMSE: 0.22554469845317865
Fold 4 - Train RMSE: 0.13876342786193638
Fold 4 - Val RMSE: 0.14204482916770125
Fold 5 - Train RMSE: 0.1377881038976042
Fold 5 - Val RMSE: 0.14177075962402227


2026/04/12 11:13:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Final RMSE: 0.12269786837216803
train_scores mean: 0.13339425739041894
train_scores std: 0.008193832881271146
val_scores mean: 0.157965966162247
val_scores std: 0.033856348904191876


2026/04/12 11:13:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run linear_regression_with_kfold at: https://dagshub.com/slomi23/slomi23_ML_Assignment_1.mlflow/#/experiments/0/runs/d935bd889de743f6badca286e070416a
🧪 View experiment at: https://dagshub.com/slomi23/slomi23_ML_Assignment_1.mlflow/#/experiments/0


Linear Regression with kfold:
train_scores mean: 0.13339425739041894
train_scores std: 0.008193832881271146
val_scores mean: 0.157965966162247
val_scores std: 0.033856348904191876


Now lets try with One-hot Encoding and Kfold

In [15]:
def preprocess_OHE(data_to_change, data_for_mapping, scaler=None):
    cat_cols = data_to_change.select_dtypes(include="object").columns
    data_to_change=fill_nas(data_to_change, data_for_mapping)
    data_to_change_dummies=pd.get_dummies(data_to_change, columns=cat_cols, drop_first=True)
    if hasattr(scaler, 'feature_names_in_'):
        data_to_change_dummies = data_to_change_dummies.reindex(
        columns=scaler.feature_names_in_, fill_value=0
    )
    return data_scale(data_to_change_dummies, scaler)
    
#lets process data for One hot Encoding and linear regression with kfold
X_train_LR_OHE = trainsetX.copy().drop(columns=["Id"])
y_train_LR_OHE = trainsetY
cat_cols = X_train_LR_OHE.select_dtypes(include="object").columns
num_cols = X_train_LR_OHE.select_dtypes(exclude="object").columns

kfold = KFold(n_splits=5, shuffle=True, random_state=12)
fold = 1
val_rmse_scores = []
train_rmse_scores = []
with mlflow.start_run(run_name="linear_regression_with_kfold_and_OHE"):
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("encoding", "OneHotEncoding")
    mlflow.log_param("scaling", "StandardScaler")
    mlflow.log_param("cv", "KFold")
    mlflow.log_param("n_splits", 5)
    mlflow.log_param("drop_first", True)
    for train_idx, val_idx in kfold.split(X_train_LR_OHE):
        X_train1 = X_train_LR_OHE.iloc[train_idx].copy()
        X_val =X_train_LR_OHE.iloc[val_idx].copy()
        y_train1 = y_train_LR_OHE.iloc[train_idx].copy()
        y_val = y_train_LR_OHE.iloc[val_idx].copy()

        X_train1_scaled, scaler = preprocess_OHE(X_train1, X_train1)
        X_val_scaled, _ = preprocess_OHE(X_val, X_train1, scaler)
        #overfittin was a problem so
        X_train1_scaled = data_rfe(X_train1_scaled, X_train1_scaled, y_train1, number_of_features=8)
        X_val_scaled = data_rfe(X_val_scaled, X_train1_scaled, y_train1, number_of_features=8)

        model = LinearRegression()
        model.fit(X_train1_scaled, y_train1)
        # predictions
        train_preds = model.predict(X_train1_scaled)
        val_preds = model.predict(X_val_scaled)

        # rmse
        train_rmse = np.sqrt(mean_squared_error(y_train1, train_preds))
        val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
        val_rmse_scores.append(val_rmse)
        train_rmse_scores.append(train_rmse)
        print(f"Fold {fold} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

        mlflow.log_metric(f"train_rmse_fold_{fold}", train_rmse)
        mlflow.log_metric(f"val_rmse_fold_{fold}", val_rmse)
        mlflow.log_params(model.get_params())
        fold += 1
    print_mean_and_std(train_rmse_scores, val_rmse_scores)

    mlflow.log_metric("val_rmse_mean", np.mean(val_rmse_scores))
    mlflow.log_metric("val_rmse_std", np.std(val_rmse_scores))
    mlflow.log_metric("train_rmse_mean", np.mean(train_rmse_scores))
    mlflow.log_metric("train_rmse_std", np.std(train_rmse_scores))

    X_full = preprocess_OHE(X_train_LR_OHE, X_train_LR_OHE)[0]

    final_model = LinearRegression()
    final_model.fit(X_full, y_train_LR_OHE)
    final_preds = final_model.predict(X_full)
    final_rmse= np.sqrt(mean_squared_error(y_train_LR_OHE, final_preds))
    mlflow.log_metric("final_rmse", final_rmse)
    print(f"Final RMSE: {final_rmse}")

    mlflow.sklearn.log_model(final_model, "linear_regression_with_kfold_and_OHE")
    mlflow.end_run()





Fold 1 | Train RMSE: 0.1869 | Val RMSE: 0.4321
Fold 2 | Train RMSE: 0.1738 | Val RMSE: 0.1864
Fold 3 | Train RMSE: 0.1881 | Val RMSE: 0.9480
Fold 4 | Train RMSE: 0.1900 | Val RMSE: 0.8053
Fold 5 | Train RMSE: 0.1827 | Val RMSE: 0.2410
train_scores mean: 0.18430683835034783
train_scores std: 0.0057701443324
val_scores mean: 0.5225535553391307
val_scores std: 0.3037794215845512


2026/04/12 11:14:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Final RMSE: 0.09464071698247319


2026/04/12 11:14:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run linear_regression_with_kfold_and_OHE at: https://dagshub.com/slomi23/slomi23_ML_Assignment_1.mlflow/#/experiments/0/runs/1588160a1024408ea812a039c419e9ce
🧪 View experiment at: https://dagshub.com/slomi23/slomi23_ML_Assignment_1.mlflow/#/experiments/0


linear regression with one hot encoding and kfold:
train_scores mean: 0.18430683835034783
train_scores std: 0.0057701443324
val_scores mean: 0.5225535553391307
val_scores std: 0.3037794215845512
(lot og overfiting since train_scores mean is really low but validation rmse score are really high so as std) 
now let's do decision tree

In [16]:
from sklearn.tree import DecisionTreeRegressor
trainset_DT=trainsetX.copy()
X_DT = trainset_DT.copy()
X_DT["SalePrice"] = trainsetY
y_DT = trainsetY

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
train_rmse_scores = []
val_rmse_scores = []
with mlflow.start_run(run_name="decision_tree_with_kfold"):
    for train_idx, val_idx in kfold.split(X_DT):

        X_train1, X_val = X_DT.iloc[train_idx], X_DT.iloc[val_idx]
        y_train1, y_val = y_DT.iloc[train_idx], y_DT.iloc[val_idx]
        X_train_raw = X_train1.copy()
        X_train1, scaler = preprocess(X_train1, X_train1)
        X_val = preprocess(X_val, X_train_raw, scaler1)[0]

        model = DecisionTreeRegressor(max_depth=6, random_state=37)        
        model.fit(X_train1, y_train1)

        preds = model.predict(X_val)
        train_rmse = np.sqrt(mean_squared_error(y_train1, model.predict(X_train1)))
        val_rmse = np.sqrt(mean_squared_error(y_val, preds))

        train_rmse_scores.append(train_rmse)
        val_rmse_scores.append(val_rmse)

        mlflow.log_metric(f"train_rmse_fold_{fold}", train_rmse)
        mlflow.log_metric(f"val_rmse_fold_{fold}", val_rmse)
        print(f"Fold {fold} - Train RMSE: {train_rmse}")
        print(f"Fold {fold} - Val RMSE: {val_rmse}")
        fold += 1
    mlflow.log_metric("val_rmse_mean", np.mean(val_rmse_scores))
    mlflow.log_metric("val_rmse_std", np.std(val_rmse_scores))

    final_model=DecisionTreeRegressor(max_depth=6, random_state=73)   
    processed_data=preprocess(X_DT, X_DT)
    final_model.fit(processed_data[0], y_DT)
    final_preds = final_model.predict(processed_data[0])
    final_rmse= np.sqrt(mean_squared_error(y_DT, final_preds))
    mlflow.log_metric("final_rmse", final_rmse)
    print(f"Final RMSE: {final_rmse}")

    mlflow.sklearn.log_model(final_model, "decision_tree_with_kfold_best2")

    print_mean_and_std(train_rmse_scores, val_rmse_scores)

Fold 1 - Train RMSE: 0.1315633090841963
Fold 1 - Val RMSE: 0.18397778658968186
Fold 2 - Train RMSE: 0.13119568556988115
Fold 2 - Val RMSE: 0.2030441953014512
Fold 3 - Train RMSE: 0.1231213761619942
Fold 3 - Val RMSE: 0.22240788781093312
Fold 4 - Train RMSE: 0.12915976122746045
Fold 4 - Val RMSE: 0.20460436317445896
Fold 5 - Train RMSE: 0.1256207344870665
Fold 5 - Val RMSE: 0.20893795831390796


2026/04/12 11:15:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Final RMSE: 0.13180198230416984


2026/04/12 11:15:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


train_scores mean: 0.12813217330611973
train_scores std: 0.0032750312980282525
val_scores mean: 0.2045944382380866
val_scores std: 0.012358275196741889
🏃 View run decision_tree_with_kfold at: https://dagshub.com/slomi23/slomi23_ML_Assignment_1.mlflow/#/experiments/0/runs/466c9e2942ec4bdab4d1e2f4b4323539
🧪 View experiment at: https://dagshub.com/slomi23/slomi23_ML_Assignment_1.mlflow/#/experiments/0


Decision tree with kfold
Final RMSE: 0.13180198230416984
train_scores mean: 0.12813217330611973
train_scores std: 0.0032750312980282525
val_scores mean: 0.2045944382380866
val_scores std: 0.01235827519674188


register the best (first) model

In [18]:
model_name="linear_regression_without_OHE_and_kfold_final"
run_id="021ad6ce146343d88dd824bcf13dab38"
model_uri=f"runs:/{run_id}/{model_name}"
with mlflow.start_run(run_name="Registration"):
    mlflow.sklearn.log_model(
        sk_model=final_model,
        artifact_path="linear_regression_without_OHE_and_kfold_final",
        registered_model_name=model_name
    )
    

2026/04/12 11:17:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 11:17:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'linear_regression_without_OHE_and_kfold_final' already exists. Creating a new version of this model...
2026/04/12 11:17:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: linear_regression_without_OHE_and_kfold_final, version 4
Created version '4' of model 'linear_regression_without_OHE_and_kfold_final'.


🏃 View run Registration at: https://dagshub.com/slomi23/slomi23_ML_Assignment_1.mlflow/#/experiments/0/runs/b6064b9f9d9a4bacbcbc9d05102d97d6
🧪 View experiment at: https://dagshub.com/slomi23/slomi23_ML_Assignment_1.mlflow/#/experiments/0
